In [0]:
# Lê da camada bronze (ingestão feita pelo notebook inicio)

df_logs = spark.table("streaming_dev.bronze_ingestao.logs_streaming")
df_catalogo = spark.table("streaming_dev.bronze_ingestao.catalogo_filmes")
df_usuarios = spark.table("streaming_dev.bronze_ingestao.usuarios")

In [0]:
from pyspark.sql import DataFrame
from pyspark.sql.functions import (
    coalesce, col, create_map, expr, lit, lower, regexp_replace, try_to_date, when
)

# ============================================
# CONSTANTES DE CONFIGURAÇÃO
# ============================================

# Thresholds de categorização (minutos)
WATCH_TIME_SHORT_THRESHOLD = 120
WATCH_TIME_LONG_THRESHOLD = 180
MINUTES_PER_HOUR = 60

# Formatos de data aceitos nos CSVs (mistos)
WATCH_DATE_FORMATS = ["yyyy-MM-dd"]
SIGNUP_DATE_FORMATS = ["yyyy-MM-dd", "MM-dd-yyyy", "dd/MM/yyyy"]

# Mapeamento de gêneros (Português -> Inglês)
GENRE_MAPPING = {
    "Terror": "Horror",
    "Animação": "Animation",
    "Documentário": "Documentary",
    "Comédia": "Comedy",
    "Ação": "Action",
}

# Mapeamento de países (variações -> padrão em inglês)
COUNTRY_MAPPING = {
    "BR": "Brazil",
    "Brasil": "Brazil",
    "USA": "United States",
    "México": "Mexico",
    "Colômbia": "Colombia",
    "Espanha": "Spain",
}

# Expressões de lookup derivadas dos mapeamentos
GENRE_MAP = create_map(*[lit(x) for par in GENRE_MAPPING.items() for x in par])
COUNTRY_MAP = create_map(*[lit(x) for par in COUNTRY_MAPPING.items() for x in par])


# ============================================
# HELPERS DE TRANSFORMAÇÃO
# ============================================

def padronizar_data(df: DataFrame, coluna: str, formatos) -> DataFrame:
    """
    Converte uma coluna de data em string para tipo date, tentando vários formatos.

    Para cada formato usa try_to_date (valores inválidos viram null em vez
    de erro) e coalesce escolhe o primeiro formato que casar.

    Args:
        df: DataFrame com a coluna de data em string
        coluna: nome da coluna a converter
        formatos: lista de padrões de data do Spark (ex.: 'yyyy-MM-dd')

    Returns:
        DataFrame com a coluna convertida para date
    """
    datas_convertidas = [try_to_date(coluna, fmt) for fmt in formatos]
    return df.withColumn(coluna, coalesce(*datas_convertidas))


def padronizar_paises(df: DataFrame) -> DataFrame:
    """
    Padroniza nomes de países para o padrão em inglês
    (BR/Brasil -> Brazil, USA -> United States, etc.).
    Países não mapeados permanecem inalterados.
    """
    return df.withColumn(
        "country",
        coalesce(COUNTRY_MAP[col("country")], col("country"))
    )


def padronizar_generos(df: DataFrame) -> DataFrame:
    """
    Padroniza nomes de gêneros de Português para Inglês.
    Gêneros não mapeados permanecem inalterados.
    """
    return df.withColumn(
        "genre",
        coalesce(GENRE_MAP[col("genre")], col("genre"))
    )


def padronizar_texto(df: DataFrame, coluna: str) -> DataFrame:
    """
    Normaliza uma coluna de texto: minúsculas e sem espaços, hífens ou
    underscores ('Smart_TV' -> 'smarttv').
    """
    return df.withColumn(
        coluna,
        regexp_replace(lower(col(coluna)), r"[\s_-]", "")
    )


def padronizar_devices(df: DataFrame) -> DataFrame:
    """
    Padroniza dispositivos para as formas canônicas
    (mobile -> Mobile, smart tv/Smart_TV -> Smart TV).
    Demais valores permanecem inalterados.
    """
    device = regexp_replace(lower(col("device")), r"[\s_-]", "")
    return df.withColumn(
        "device",
        when(device == "mobile", "Mobile")
        .when(device == "smarttv", "Smart TV")
        .otherwise(col("device"))
    )


# ============================================
# TRANSFORMAÇÕES POR DATAFRAME
# ============================================

def transformar_logs(df: DataFrame) -> DataFrame:
    """
    Aplica transformações e limpeza nos logs de streaming.

    Transformações realizadas:
    - Padroniza playback_status para lowercase
    - Converte watch_date para date
    - Converte watch_time_minutes para numérico (trata valor 'zero')
    - Calcula watch_time_hours e categoriza o tempo (Muito longo/Longo/Curto)
    - Padroniza dispositivos, países e subscription_type
    """
    return (
        df
        .withColumn("playback_status", lower(col("playback_status")))
        .transform(lambda d: padronizar_data(d, "watch_date", WATCH_DATE_FORMATS))
        .withColumn(
            "watch_time_minutes",
            expr("try_cast(replace(watch_time_minutes, 'zero', '0') AS DOUBLE)")
        )
        .withColumn("watch_time_hours", col("watch_time_minutes") / MINUTES_PER_HOUR)
        .withColumn(
            "watch_categories",
            when(col("watch_time_minutes") > WATCH_TIME_LONG_THRESHOLD, "Muito longo")
            .when(col("watch_time_minutes") >= WATCH_TIME_SHORT_THRESHOLD, "Longo")
            .otherwise("Curto")
        )
        .transform(padronizar_devices)
        .transform(padronizar_paises)
        .transform(lambda d: padronizar_texto(d, "subscription_type"))
    )


def transformar_catalogo(df: DataFrame) -> DataFrame:
    """
    Padroniza o catálogo de filmes:
    - Gêneros de Português para Inglês
    - content_type em minúsculas, sem espaços/hífens/underscores
    """
    return (
        df
        .transform(padronizar_generos)
        .transform(lambda d: padronizar_texto(d, "content_type"))
    )


def transformar_usuarios(df: DataFrame) -> DataFrame:
    """
    Padroniza os dados de usuários:
    - signup_date para date (formatos mistos: yyyy-MM-dd, MM-dd-yyyy, dd/MM/yyyy)
    - Nomes de países
    - subscription_type em minúsculas, sem espaços/hífens/underscores
    """
    return (
        df
        .transform(lambda d: padronizar_data(d, "signup_date", SIGNUP_DATE_FORMATS))
        .transform(padronizar_paises)
        .transform(lambda d: padronizar_texto(d, "subscription_type"))
    )

In [0]:
# Aplicar transformações (uma função por DataFrame)
df_logs = transformar_logs(df_logs)
df_catalogo = transformar_catalogo(df_catalogo)
df_usuarios = transformar_usuarios(df_usuarios)


In [0]:
# Salvando os DataFrames nas tabelas silver
TABELAS_SILVER = {
    "streaming_dev.silver.logs_streaming": df_logs,
    "streaming_dev.silver.catalogo_filmes": df_catalogo,
    "streaming_dev.silver.usuarios": df_usuarios,
}

for tabela, df in TABELAS_SILVER.items():
    df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(tabela)
